In [5]:
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.feature_extraction.text import CountVectorizer
import joblib
import os

In [ ]:
file_path = r'D:\TTS_ITMO\tts_labs_itmo\labs\lab1_text\data\dev_sentences.csv'

with open(file_path, 'r', encoding='utf-8') as f:
    for i in range(10):
        line = f.readline()
        print(f"Строка {i+1}: {repr(line)}")

In [10]:
with open(file_path, 'r', encoding='utf-8') as f:
    for i in range(10):
        line = f.readline()
        print(f"Строка {i+1}: {repr(line)}")

Строка 1: 'text|is_normalized\n'
Строка 2: 'Сегодня обещают около нуля и ветер.|1\n'
Строка 3: 'Доставка займёт два-три дня.|1\n'
Строка 4: 'Так как заявка уже принята, менять ничего не нужно.|1\n'
Строка 5: 'Николай Николаевич, повторите, пожалуйста, свой вопрос.|1\n'
Строка 6: 'Договор заключён восьмого марта две тысячи двадцать четвёртого года.|1\n'
Строка 7: 'Температура за окном около нуля.|1\n'
Строка 8: 'Скажите, пожалуйста, что именно не работает?((|0\n'
Строка 9: 'Он приехал из Санкт-Петербурга вчера вечером???|0\n'
Строка 10: 'Заявка будет рассмотрена в 3-й рабочий день.|0\n'


In [16]:
dev_data = pd.read_csv(
    file_path,
    sep = '|',
    encoding = 'utf-8'
)

In [17]:
dev_data.head(10)

,text,is_normalized
0,Сегодня обещают около нуля и ветер.,1
1,Доставка займёт два-три дня.,1
2,"Так как заявка уже принята, менять ничего не н...",1
3,"Николай Николаевич, повторите, пожалуйста, сво...",1
4,Договор заключён восьмого марта две тысячи два...,1
5,Температура за окном около нуля.,1
6,"Скажите, пожалуйста, что именно не работает?((",0
7,Он приехал из Санкт-Петербурга вчера вечером???,0
8,Заявка будет рассмотрена в 3-й рабочий день.,0
9,"Дверь была приоткрыта, но входить почему-то не...",1


In [18]:
dev_data.describe

<bound method NDFrame.describe of                                                   text  is_normalized
0                  Сегодня обещают около нуля и ветер.              1
1                         Доставка займёт два-три дня.              1
2    Так как заявка уже принята, менять ничего не н...              1
3    Николай Николаевич, повторите, пожалуйста, сво...              1
4    Договор заключён восьмого марта две тысячи два...              1
..                                                 ...            ...
295          Температура за окном минус пять градусов.              1
296           Ваш Redmi Note 12 поддерживает сеть NFC.              0
297             Ваш номер 8-800-974-66-83 подтверждён.              0
298      Оплата спишется первого числа каждого месяца.              1
299  Она поставила чашку на стол так осторожно, сло...              1

[300 rows x 2 columns]>

In [19]:
#Фильтр
import re
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
import joblib
import os

class TextFilter:
    """
    Фильтр для определения нормализованности текста.
    """
    
    def __init__(self):
        self.model = None
        self.feature_names = None
        self.is_fitted = False
        
        # Стоп-слова для исключения из сокращений
        self.stop_words = [
            "он", "ты", "мы", "вы", "я", "она", "они", "то", "все", "его", "ее", "же", "бы", "да", "их", "ей", "уж", "ну",
            "мне", "те", "се", "вас", "нас", "сам", "ах", "ох", "эх", "ой", "ай", "эт", "ли",
            "меня", "тебя", "себя", "вами", "нами", "мой", "твой", "свой",
            "это", "эти", "эта", "тот", "та", "те", "вон", "вот",
            "кто", "что", "чей", "чья", "чьё",
            "весь", "вся", "всё", "сама", "сами",
            "ага", "увы", "ого"
        ]
    
    def _find_abbr_with_filter(self, text):
        """Находит сокращения, исключая стоп-слова"""
        if not isinstance(text, str):
            text = str(text)
        
        candidates = re.findall(r'\b[а-яА-Я]{1,2}\.', text)
        
        real_abbr = []
        for word in candidates:
            word_without_dot = word[:-1].lower()
            if word_without_dot not in self.stop_words:
                real_abbr.append(word)
        
        return real_abbr
    
    def _find_true_acronyms(self, text):
        """Находит настоящие аббревиатуры (без гласных)"""
        if not isinstance(text, str):
            text = str(text)
        
        words = re.findall(r'\b[А-ЯЁ]{2,}\b', text)
        vowels = 'АЕЁИОУЫЭЮЯ'
        return [w for w in words if not any(v in w for v in vowels)]
    
    def _extract_features(self, text):
        """
        Извлекает признаки из текста.
        
        Признаки основаны на анализе корпуса RUSLAN:
        - Сокращения (с фильтром стоп-слов)
        - Настоящие аббревиатуры (без гласных)
        - Технические символы
        - Цифры
        - Знаки препинания
        - Длина текста
        """
        if not isinstance(text, str):
            text = str(text)
        
        features = {}
        
        # 1. Сокращения (с фильтром стоп-слов)
        abbrs = self._find_abbr_with_filter(text)
        features['has_abbr'] = int(len(abbrs) > 0)
        features['abbr_count'] = len(abbrs)
        
        # 2. Настоящие аббревиатуры (без гласных)
        true_acronyms = self._find_true_acronyms(text)
        features['has_acronym'] = int(len(true_acronyms) > 0)
        features['acronym_count'] = len(true_acronyms)
        
        # 3. Технические символы
        tech_chars = re.findall(r'[\(\)\[\]\{\}\/\*<>]', text)
        features['has_tech'] = int(len(tech_chars) > 0)
        features['tech_count'] = len(tech_chars)
        
        # 4. Цифры
        digits = re.findall(r'\d', text)
        features['has_digits'] = int(len(digits) > 0)
        features['digit_count'] = len(digits)
        
        # 5. Латиница
        latin = re.findall(r'[a-zA-Z]', text)
        features['has_latin'] = int(len(latin) > 0)
        features['latin_count'] = len(latin)
        
        # 6. Кавычки
        quotes = re.findall(r'[«»“”„]', text)
        features['has_quotes'] = int(len(quotes) > 0)
        features['quote_count'] = len(quotes)
        
        # 7. Тире
        dashes = re.findall(r'[‑–—]', text)
        features['has_dashes'] = int(len(dashes) > 0)
        features['dash_count'] = len(dashes)
        
        # 8. Эмодзи
        emoji_pattern = re.compile("["
            u"\U0001F600-\U0001F64F"
            u"\U0001F300-\U0001F5FF"
            u"\U0001F680-\U0001F6FF"
            u"\U00002702-\U000027B0"
            u"\U000024C2-\U0001F251"
            "]+", flags=re.UNICODE)
        emojis = emoji_pattern.findall(text)
        features['has_emoji'] = int(len(emojis) > 0)
        
        # 9. Знаки препинания
        features['has_exclamation'] = int('!' in text)
        features['has_question'] = int('?' in text)
        features['has_multiple_punct'] = int(bool(re.search(r'[!?]{2,}', text)))
        features['has_ellipsis'] = int(bool(re.search(r'…', text)))
        features['has_brackets'] = int(bool(re.search(r'[\(\)]', text)))
        
        # 10. Доля "плохих" символов
        bad_chars = re.findall(r'[^а-яА-ЯёЁ\s\.\,\!\?\-]', text)
        features['bad_ratio'] = len(bad_chars) / len(text) if len(text) > 0 else 0
        features['bad_count'] = len(bad_chars)
        
        # 11. Длина текста и количество слов
        features['text_length'] = len(text)
        features['word_count'] = len(text.split())
        features['sentence_count'] = len(re.findall(r'[.!?]+', text))
        
        return pd.Series(features)
    
    def fit(self, texts, labels):
        """Обучает модель"""
        print("Извлечение признаков...")
        X = pd.DataFrame([self._extract_features(t) for t in texts])
        self.feature_names = X.columns.tolist()
        
        print(f"✅ Извлечено {len(self.feature_names)} признаков:")
        for i, name in enumerate(self.feature_names, 1):
            print(f"   {i}. {name}")
        
        print("\nОбучение модели...")
        self.model = RandomForestClassifier(
            n_estimators=100,
            max_depth=10,
            min_samples_split=5,
            class_weight='balanced',
            random_state=42
        )
        self.model.fit(X, labels)
        self.is_fitted = True
        print("✅ Модель обучена!")
        
        # Важность признаков
        importance = pd.DataFrame({
            'feature': self.feature_names,
            'importance': self.model.feature_importances_
        }).sort_values('importance', ascending=False)
        
        print("\n📊 Топ-10 важных признаков:")
        for _, row in importance.head(10).iterrows():
            print(f"   {row['feature']}: {row['importance']:.4f}")
        
        return self
    
    def predict(self, texts):
        """Предсказывает класс"""
        if not self.is_fitted:
            raise ValueError("Модель не обучена!")
        
        X = pd.DataFrame([self._extract_features(t) for t in texts])
        return self.model.predict(X)
    
    def predict_proba(self, texts):
        """Возвращает вероятности"""
        if not self.is_fitted:
            raise ValueError("Модель не обучена!")
        
        X = pd.DataFrame([self._extract_features(t) for t in texts])
        return self.model.predict_proba(X)
    
    def save(self, path='labs/lab1_text/model.pkl'):
        """Сохраняет модель"""
        if not self.is_fitted:
            raise ValueError("Модель не обучена!")
        
        os.makedirs(os.path.dirname(path), exist_ok=True)
        joblib.dump({
            'model': self.model,
            'feature_names': self.feature_names,
            'is_fitted': self.is_fitted,
            'stop_words': self.stop_words
        }, path)
        print(f"✅ Модель сохранена в {path}")
    
    def load(self, path='labs/lab1_text/model.pkl'):
        """Загружает модель"""
        data = joblib.load(path)
        self.model = data['model']
        self.feature_names = data['feature_names']
        self.is_fitted = data['is_fitted']
        if 'stop_words' in data:
            self.stop_words = data['stop_words']
        print(f"✅ Модель загружена из {path}")
        return self

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix

print("=" * 80)
print("ОБУЧЕНИЕ ФИЛЬТРА С ВАШИМИ ФУНКЦИЯМИ")
print("=" * 80)

# Загружаем данные
dev_data = pd.read_csv(file_path, sep='|')
print(f"Загружено {len(dev_data)} предложений")

# Разделяем на train/validation
X_train, X_val, y_train, y_val = train_test_split(
    dev_data['text'],
    dev_data['is_normalized'],
    test_size=0.2,
    random_state=42,
    stratify=dev_data['is_normalized']
)

print(f"\nОбучающая выборка: {len(X_train)}")
print(f"Валидационная выборка: {len(X_val)}")

# Обучаем фильтр
filter = TextFilter()
filter.fit(X_train, y_train)



ОБУЧЕНИЕ ФИЛЬТРА С ВАШИМИ ФУНКЦИЯМИ
Загружено 300 предложений

Обучающая выборка: 240
Валидационная выборка: 60
Извлечение признаков...
✅ Извлечено 25 признаков:
   1. has_abbr
   2. abbr_count
   3. has_acronym
   4. acronym_count
   5. has_tech
   6. tech_count
   7. has_digits
   8. digit_count
   9. has_latin
   10. latin_count
   11. has_quotes
   12. quote_count
   13. has_dashes
   14. dash_count
   15. has_emoji
   16. has_exclamation
   17. has_question
   18. has_multiple_punct
   19. has_ellipsis
   20. has_brackets
   21. bad_ratio
   22. bad_count
   23. text_length
   24. word_count
   25. sentence_count

Обучение модели...
✅ Модель обучена!

📊 Топ-10 важных признаков:
   bad_ratio: 0.2073
   bad_count: 0.1335
   text_length: 0.0887
   has_digits: 0.0838
   digit_count: 0.0756
   latin_count: 0.0672
   has_latin: 0.0515
   word_count: 0.0510
   abbr_count: 0.0363
   has_exclamation: 0.0362

✅ F1-score на валидации: 0.8788

📊 Отчет по классификации:
                   prec

In [24]:
# Проверяем качество
y_pred = filter.predict(X_val)
f1 = f1_score(y_val, y_pred)

print(f"\n✅ F1-score на валидации: {f1:.4f}")

print("\n📊 Отчет по классификации:")
print(classification_report(y_val, y_pred, target_names=['ненормализованные', 'нормализованные']))

print("\n📊 Матрица ошибок:")
print(confusion_matrix(y_val, y_pred))

# Сохраняем модель
filter.save('labs/lab1_text/model.pkl')

print("\n" + "=" * 80)
print("✅ Обучение завершено!")


✅ F1-score на валидации: 0.8788

📊 Отчет по классификации:
                   precision    recall  f1-score   support

ненормализованные       1.00      0.74      0.85        31
  нормализованные       0.78      1.00      0.88        29

         accuracy                           0.87        60
        macro avg       0.89      0.87      0.87        60
     weighted avg       0.90      0.87      0.86        60


📊 Матрица ошибок:
[[23  8]
 [ 0 29]]
✅ Модель сохранена в labs/lab1_text/model.pkl

✅ Обучение завершено!


In [ ]:

# Проверяем качество
y_pred = filter.predict(X_val)
f1 = f1_score(y_val, y_pred)

print(f"\n✅ F1-score на валидации: {f1:.4f}")

print("\n📊 Отчет по классификации:")
print(classification_report(y_val, y_pred, target_names=['ненормализованные', 'нормализованные']))

print("\n📊 Матрица ошибок:")
print(confusion_matrix(y_val, y_pred))

# Сохраняем модель
filter.save('labs/lab1_text/model.pkl')

print("\n" + "=" * 80)
print("✅ Обучение завершено!")

In [22]:
print("=" * 80)
print("ПРОВЕРКА НА ТЕСТОВЫХ ПРИМЕРАХ")
print("=" * 80)

# Загружаем модель
filter = TextFilter()
filter.load('labs/lab1_text/model.pkl')

# Тестовые примеры
test_examples = [
    "Сегодня обещают около нуля и ветер.",
    "Скажите, пожалуйста, что именно не работает? ((",
    "Он приехал из Санкт-Петербурга вчера вечером???",
    "Заявка будет рассмотрена в 3-й рабочий день.",
    "Договор заключён восьмого марта две тысячи двадцать четвёртого года.",
    "Я учусь в МГУ.",
    "ул. Пушкина, д. 10",
    "Привет! Как дела?",
]

print("\n📌 Результаты:")
print("-" * 80)

for text in test_examples:
    pred = filter.predict([text])[0]
    proba = filter.predict_proba([text])[0]
    status = "✅ НОРМАЛИЗОВАН" if pred == 1 else "❌ НЕНОРМАЛИЗОВАН"
    print(f"{status} | {text[:50]:50} | proba: {proba[1]:.3f}")

ПРОВЕРКА НА ТЕСТОВЫХ ПРИМЕРАХ
✅ Модель загружена из labs/lab1_text/model.pkl

📌 Результаты:
--------------------------------------------------------------------------------
✅ НОРМАЛИЗОВАН | Сегодня обещают около нуля и ветер.                | proba: 0.811
❌ НЕНОРМАЛИЗОВАН | Скажите, пожалуйста, что именно не работает? ((    | proba: 0.099
✅ НОРМАЛИЗОВАН | Он приехал из Санкт-Петербурга вчера вечером???    | proba: 0.512
❌ НЕНОРМАЛИЗОВАН | Заявка будет рассмотрена в 3-й рабочий день.       | proba: 0.047
✅ НОРМАЛИЗОВАН | Договор заключён восьмого марта две тысячи двадцат | proba: 0.969
✅ НОРМАЛИЗОВАН | Я учусь в МГУ.                                     | proba: 0.767
❌ НЕНОРМАЛИЗОВАН | ул. Пушкина, д. 10                                 | proba: 0.000
❌ НЕНОРМАЛИЗОВАН | Привет! Как дела?                                  | proba: 0.480
